# Notebook 03 — Anomaly Detection

**Goal:** Train and evaluate two anomaly detection architectures on UCSD Ped1 / Ped2:
1. **ConvAE** — reconstruction-error based frame-level detector
2. **FutureFrameNet** — U-Net future-frame predictor (Liu et al. CVPR 2018 variant)

**Evaluation:** AUC-ROC, EER, Average Precision, decision visualisation with heatmaps.

> Models are trained only on normal frames. Anomaly score = reconstruction/prediction error.

In [ ]:
# Auto-detect repo root for JupyterLab / GCP / local execution

from pathlib import Path

import sys



def find_repo_root(start=None):

    start_path = Path(start or Path.cwd()).resolve()

    for candidate in [start_path, *start_path.parents]:

        if (candidate / 'src').exists() and (candidate / 'notebooks').exists():

            return candidate

    return start_path



REPO_ROOT = find_repo_root()

DATA_ROOT = REPO_ROOT / 'data'

CKPT_ROOT = REPO_ROOT / 'checkpoints'

EXPS_ROOT = REPO_ROOT / 'experiments'

EXPS_ROOT.mkdir(exist_ok=True)

CKPT_ROOT.mkdir(exist_ok=True)

sys.path.insert(0, str(REPO_ROOT))



import torch

import torch.optim as optim

import numpy as np

import matplotlib.pyplot as plt

from sklearn.metrics import roc_curve, auc



from src.data_loaders import get_ucsd_loaders

from src.models import ConvAE, FutureFrameNet

from src.training import AnomalyTrainer

from src.evaluation import evaluate_anomaly_detection

from src.utils import show_anomaly_heatmap



DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Device: {DEVICE}')

if torch.cuda.is_available():

    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ── Cell 2: Configuration ─────────────────────────────────────────────────────
CFG = {
    'ped'           : 'ped2',      # 'ped1' or 'ped2'
    'img_size'      : (128, 192),  # (H, W)
    'clip_len'      : 8,           # frames per clip
    'batch_size'    : 16,
    'workers'       : 2,
    'epochs_convae' : 100,
    'epochs_ffnet'  : 80,
    'lr'            : 1e-3,
    'weight_decay'  : 1e-5,
    'patience'      : 15,
}

if DEVICE == 'cpu':
    CFG['epochs_convae'] = 3
    CFG['epochs_ffnet']  = 3
    CFG['batch_size']    = 4
    print('CPU mode: reduced epochs for demo run')

print('Configuration:', CFG)

In [ ]:
# ── Cell 3: Load UCSD dataset ─────────────────────────────────────────────────
print(f'Loading UCSD {CFG["ped"]}...')
train_loader, test_loader = get_ucsd_loaders(
    data_root=str(DATA_ROOT),
    ped=CFG['ped'],
    img_size=CFG['img_size'],
    batch_size=CFG['batch_size'],
    clip_len=CFG['clip_len'],
    num_workers=CFG['workers'],
)

print(f'Train batches: {len(train_loader)} ({len(train_loader.dataset)} clips — normal only)')
print(f'Test  batches: {len(test_loader)} ({len(test_loader.dataset)} clips)')

frames = next(iter(train_loader))
if isinstance(frames, (list, tuple)):
    frames = frames[0]
print(f'Clip tensor shape: {tuple(frames.shape)}   (B, T, C, H, W)')

In [ ]:
# ── Cell 4: Visualise UCSD sample clips ───────────────────────────────────────
fig, axes = plt.subplots(2, CFG['clip_len'], figsize=(CFG['clip_len'] * 2, 5))
fig.suptitle(f'UCSD {CFG["ped"].upper()} — Normal Training Clips (2 samples)', fontsize=12)

train_iter = iter(train_loader)
for row in range(2):
    batch = next(train_iter)
    if isinstance(batch, (list, tuple)):
        batch = batch[0]
    clip = batch[0].numpy()  # (T, C, H, W)
    for col in range(min(CFG['clip_len'], len(clip))):
        frame = clip[col]
        if frame.shape[0] == 1:
            axes[row, col].imshow(frame[0], cmap='gray', vmin=0, vmax=1)
        else:
            axes[row, col].imshow(frame.transpose(1,2,0).clip(0,1))
        axes[row, col].axis('off')
        axes[row, col].set_title(f't={col}', fontsize=7)

plt.tight_layout()
plt.savefig(EXPS_ROOT / f'anomaly_{CFG["ped"]}_samples.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 5: Train ConvAE ──────────────────────────────────────────────────────
print('=' * 60)
print('TRAINING ConvAE')
print('=' * 60)

in_ch = 1  # grayscale
model_convae = ConvAE(in_channels=in_ch, base_ch=32).to(DEVICE)
print(f'ConvAE parameters: {sum(p.numel() for p in model_convae.parameters()):,}')

opt_cae   = optim.Adam(model_convae.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
sched_cae = optim.lr_scheduler.StepLR(opt_cae, step_size=30, gamma=0.5)

trainer_cae = AnomalyTrainer(
    model=model_convae, optimizer=opt_cae, scheduler=sched_cae, device=DEVICE,
    experiment_name=f'convae_{CFG["ped"]}',
    save_dir=str(CKPT_ROOT), log_dir=str(REPO_ROOT / 'runs'),
)
trainer_cae.load_checkpoint('last.pt')

history_cae = trainer_cae.train(
    train_loader, test_loader,
    epochs=CFG['epochs_convae'], patience=CFG['patience'],
    metric_key='auc',
    lower_is_better=False,
)

In [ ]:
# ── Cell 6: Train FutureFrameNet ──────────────────────────────────────────────
print('=' * 60)
print('TRAINING FutureFrameNet')
print('=' * 60)

model_ffnet = FutureFrameNet(
    num_input_frames=CFG['clip_len'] - 1,
    in_channels=in_ch,
    base_ch=32,
).to(DEVICE)
print(f'FutureFrameNet parameters: {sum(p.numel() for p in model_ffnet.parameters()):,}')

opt_ffn   = optim.Adam(model_ffnet.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
sched_ffn = optim.lr_scheduler.CosineAnnealingLR(opt_ffn, T_max=CFG['epochs_ffnet'], eta_min=1e-6)

trainer_ffn = AnomalyTrainer(
    model=model_ffnet, optimizer=opt_ffn, scheduler=sched_ffn, device=DEVICE,
    experiment_name=f'ffnet_{CFG["ped"]}',
    save_dir=str(CKPT_ROOT), log_dir=str(REPO_ROOT / 'runs'),
)
trainer_ffn.load_checkpoint('last.pt')

history_ffn = trainer_ffn.train(
    train_loader, test_loader,
    epochs=CFG['epochs_ffnet'], patience=CFG['patience'],
    metric_key='auc',
    lower_is_better=False,
)

In [ ]:
# ── Cell 7: Evaluate both models ──────────────────────────────────────────────
trainer_cae.load_checkpoint('best.pt')
trainer_ffn.load_checkpoint('best.pt')

cae_results = evaluate_anomaly_detection(model_convae, train_loader, test_loader, DEVICE)
ffn_results = evaluate_anomaly_detection(model_ffnet,  train_loader, test_loader, DEVICE)

print(f'\nConvAE on UCSD {CFG["ped"]}      →  AUC: {cae_results["auc"]:.2f}%  EER: {cae_results["eer"]:.2f}%  AP: {cae_results["ap"]:.2f}%')
print(f'FutureFrameNet on UCSD {CFG["ped"]} →  AUC: {ffn_results["auc"]:.2f}%  EER: {ffn_results["eer"]:.2f}%  AP: {ffn_results["ap"]:.2f}%')

In [ ]:
# ── Cell 8: ROC curve comparison ──────────────────────────────────────────────
from src.evaluation.anomaly_metrics import _collect_scores

fig, ax = plt.subplots(figsize=(7, 6))

for name, model, trainer, color in [
    ('ConvAE',          model_convae, trainer_cae, 'blue'),
    ('FutureFrameNet',  model_ffnet,  trainer_ffn, 'red'),
]:
    trainer.load_checkpoint('best.pt')
    try:
        scores, labels = _collect_scores(model, test_loader, DEVICE)
        fpr, tpr, _ = roc_curve(labels, scores)
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=color, label=f"{name} (AUC={roc_auc:.3f})")
    except Exception as e:
        print(f'Could not plot ROC for {name}: {e}')

ax.plot([0,1],[0,1],'k--', alpha=0.5, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title(f'ROC Curves — UCSD {CFG["ped"].upper()}')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(EXPS_ROOT / f'anomaly_{CFG["ped"]}_roc.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 9: Anomaly heatmap visualisation ─────────────────────────────────────
import torch.nn.functional as F

model_convae.eval()
model_ffnet.eval()

test_iter = iter(test_loader)
n_vis = 3

for sample_idx in range(n_vis):
    batch = next(test_iter)
    if isinstance(batch, (list, tuple)):
        clips, labels = batch
    else:
        clips = batch
        labels = torch.zeros(clips.shape[0])

    # Use the first clip in the batch
    clip  = clips[0].unsqueeze(0).to(DEVICE)  # (1, T, C, H, W)
    label = labels[0].item()

    # For ConvAE: reconstruct middle frame
    mid = CFG['clip_len'] // 2
    frame_b = clip[:, mid]  # (1, C, H, W)
    with torch.no_grad():
        recon = model_convae(frame_b)
    err_map = (frame_b - recon).abs().squeeze().cpu().numpy()
    if err_map.ndim == 3:
        err_map = err_map[0]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    anom_label = 'ANOMALOUS' if label == 1 else 'NORMAL'
    fig.suptitle(f'Sample {sample_idx+1} — {anom_label}', fontsize=12,
                 color='red' if label == 1 else 'green')

    frame_np = frame_b.squeeze().cpu().numpy()
    axes[0].imshow(frame_np, cmap='gray'); axes[0].set_title('Input Frame'); axes[0].axis('off')
    axes[1].imshow(recon.squeeze().cpu().numpy(), cmap='gray'); axes[1].set_title('Reconstruction'); axes[1].axis('off')
    im = axes[2].imshow(err_map, cmap='hot'); axes[2].set_title('Error Heatmap'); axes[2].axis('off')
    plt.colorbar(im, ax=axes[2], fraction=0.046)

    plt.tight_layout()
    plt.savefig(EXPS_ROOT / f'anomaly_heatmap_{sample_idx+1}.png', dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()

In [ ]:
# ── Cell 10: Save comparison table ───────────────────────────────────────────
from src.utils import make_results_table

all_results = {
    'ConvAE'         : cae_results,
    'FutureFrameNet' : ffn_results,
}

table = make_results_table(all_results, columns=['auc', 'eer', 'ap', 'f1'])
print(f'\n=== UCSD {CFG["ped"].upper()} Anomaly Detection Results ===')
print(table)

(EXPS_ROOT / f'anomaly_{CFG["ped"]}_results.md').write_text(
    f'# UCSD {CFG["ped"].upper()} Anomaly Detection Results\n\n' + table
)
print('Saved to experiments/')

In [ ]:
# ── Cell 11: Optional — run on Ped1 as well ───────────────────────────────────
other_ped = 'ped1' if CFG['ped'] == 'ped2' else 'ped2'

other_train_dir = DATA_ROOT / 'UCSD_Anomaly_Dataset.v1p2' / f'UCSD{other_ped}' / 'Train'

if other_train_dir.exists():
    print(f'Running ConvAE on {other_ped}...')
    other_train, other_test = get_ucsd_loaders(
        data_root=str(DATA_ROOT),
        ped=other_ped,
        img_size=CFG['img_size'],
        batch_size=CFG['batch_size'],
        clip_len=CFG['clip_len'],
        num_workers=CFG['workers'],
    )
    m2 = ConvAE(in_channels=in_ch, base_ch=32).to(DEVICE)
    o2 = optim.Adam(m2.parameters(), lr=CFG['lr'])
    t2 = AnomalyTrainer(
        model=m2, optimizer=o2, device=DEVICE,
        experiment_name=f'convae_{other_ped}',
        save_dir=str(CKPT_ROOT), log_dir=str(REPO_ROOT / 'runs'),
    )
    t2.load_checkpoint('last.pt')
    t2.train(other_train, other_test, epochs=CFG['epochs_convae'], patience=CFG['patience'],
             metric_key='auc', lower_is_better=False)
    t2.load_checkpoint('best.pt')
    r2 = evaluate_anomaly_detection(m2, other_train, other_test, DEVICE)
    print(f'ConvAE {other_ped} → AUC: {r2["auc"]:.2f}%  EER: {r2["eer"]:.2f}%')
else:
    print(f'{other_ped} data not found at {other_train_dir}, skipping.')

---
## Anomaly Detection Complete!

Results saved to `experiments/`. Continue with **04_tracking.ipynb**.